In [44]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document
import pandas as pd
import pickle
        
from omegaconf import OmegaConf
config = OmegaConf.load("./params.yaml")


data = pickle.load(open(config.data.pr_data_unstemmed_save_path, 'rb'))
data = pd.DataFrame(data)

data['tagged_description'] = (data['id'].astype(str) + ' ' + data['tags'])

documents = [Document(page_content=text) for text in data['tagged_description']]

text_splitter = CharacterTextSplitter(chunk_size=0, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(documents)

In [81]:
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma  import Chroma
from dotenv import load_dotenv

load_dotenv()

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

persist_dir = config.data.documentstore

ids = [str(x).split("=")[1].split()[0].strip("'") for x in documents]


db_movies = Chroma.from_documents(
    documents,
    collection_name= 'TMDB5000',
    ids=ids,
    embedding=embedding_model,
    persist_directory=persist_dir,
    create_collection_if_not_exists=True
)


In [86]:
from langchain_chroma  import Chroma

# Reload the persistent Chroma database
vectorstore = Chroma(
    embedding_function=embedding_model,
    persist_directory=persist_dir
)

all_items = vectorstore.get(ids=ids)

In [9]:
ix = data[data['title'] == 'Hulk'].index[0]
data.iloc[ix, 2]
j = db_movies.similarity_search_with_score(query=data.iloc[ix, 2], k=6)


In [130]:
k = pd.DataFrame(j)

k.iloc[:, 0] = k.iloc[:, 0].apply(lambda x: int(str(x).split("=")[1].split()[0].strip("'")))

scores = k.set_index(keys=0).T.iloc[:, 1:].to_dict(orient='records')[0]

In [133]:
scores.keys()

dict_keys([1724, 36658, 38055, 246655, 15239])

In [132]:
data[data['id'].isin(list(scores.keys()))]

,id,title,tags,tagged_description
64,246655,X-Men: Apocalypse,sciencefiction after the re-emergence of the w...,246655 sciencefiction after the re-emergence o...
174,1724,The Incredible Hulk,sciencefiction action adventure scientist bruc...,1724 sciencefiction action adventure scientist...
196,38055,Megamind,animation action comedy family sciencefiction ...,38055 animation action comedy family sciencefi...
203,36658,X2,adventure action sciencefiction thriller profe...,36658 adventure action sciencefiction thriller...
4569,15239,The Toxic Avenger,sciencefiction action comedy horror tromaville...,15239 sciencefiction action comedy horror trom...


In [168]:
recommend('Hulk', 5, data)

{'status_code': 7, 'status_message': 'Invalid API key: You must be granted a valid key.', 'success': False}


KeyError: 'poster_path'

In [174]:

load_dotenv()
api_key = os.getenv("TMDB_API_KEY")

In [173]:
import os
import requests

load_dotenv()
api_key = os.getenv("TMDB_API_KEY")


def fetch_poster(movie_id):
    api_key = '<API_KEY>'
    response = requests.get(f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}")
    data = response.json()
    print(data)
    # if not data['success']:
    #     return -1
    # else:
    poster_path = data['poster_path']
    full_path = "https://image.tmdb.org/t/p/w500" + poster_path
    return full_path

In [162]:
def recommend(option, n, df):
    ix = df[df['title'] == option].index[0]
    df.iloc[ix, 2]
    j = db_movies.similarity_search_with_score(query=df.iloc[ix, 2], k=n+1)
    k = pd.DataFrame(j)
    k.iloc[:, 0] = k.iloc[:, 0].apply(lambda x: int(str(x).split("=")[1].split()[0].strip("'")))
    list_of_recommendations = k.set_index(keys=0).T.iloc[:, 1:].to_dict(orient='records')[0]

    counter = 1
    top_n = []
    top_n_posters = []
    for movie_idx in list_of_recommendations.keys():
        if counter < n + 1:
            counter += 1
            top_n.append(df[df['id'] == movie_idx].title.values[0])
            top_n_posters.append(fetch_poster(df[df['id'] == movie_idx].title.values[0]))
        else:
            break
    
    return top_n
    # st.write(f'Here are {n} movies that are most similar to {option}')
    # col1, col2, col3, col4, col5 = st.columns(5)
    # with col1:
    #     st.text(top_n[0])
    #     st.image(top_n_posters[0])
    # with col2:
    #     st.text(top_n[1])
    #     st.image(top_n_posters[1])
    # with col3:
    #     st.text(top_n[2])
    #     st.image(top_n_posters[2])
    # with col4:
    #     st.text(top_n[3])
    #     st.image(top_n_posters[3])
    # with col5:
    #     st.text(top_n[4])
    #     st.image(top_n_posters[4])